# Installation

In [1]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

# Model

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = False # Use 4bit quantization to reduce memory usage. Can be False.


model, tokenizer =  FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct", # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.7.1: Fast Llama patching. Transformers: 4.53.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.7.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [5]:
model.print_trainable_parameters()

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


# Data Prep

In [7]:
import pandas as pd
import json

def format_for_llama3_2(row):
    """
    Formats a single row of data into the Llama 3.2 chat template.

    Args:
        row (pd.Series): A row from a DataFrame with 'system', 'user',
                         and 'assistant' columns.

    Returns:
        dict: A dictionary containing the formatted chat string or None if a row is invalid.
    """
    # Llama 3.2 Chat Template Structure
    # <|begin_of_text|><|start_header_id|>system<|end_header_id|>
    #
    # {system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>
    #
    # {user_message}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
    #
    # {assistant_response}<|eot_id|>

    system_prompt = row.get('system', '')
    user_message = row.get('user', '')
    assistant_response = row.get('assistant', '')

    # Start of the conversation
    formatted_chat = "<|begin_of_text|>"

    # Add system prompt if it exists and is not empty
    if pd.notna(system_prompt) and str(system_prompt).strip():
        formatted_chat += f"<|start_header_id|>system<|end_header_id|>\n\n{system_prompt}<|eot_id|>"

    # Add user message
    # This part is mandatory for a valid conversation turn
    if pd.notna(user_message) and str(user_message).strip():
        formatted_chat += f"<|start_header_id|>user<|end_header_id|>\n\n{user_message}<|eot_id|>"
    else:
        # User message is essential. Skip if it's missing.
        print(f"Skipping row due to missing user message: {row.name}")
        return None

    # Add assistant response
    # This part is also mandatory for a training example
    if pd.notna(assistant_response) and str(assistant_response).strip():
        formatted_chat += f"<|start_header_id|>assistant<|end_header_id|>\n\n{assistant_response}<|eot_id|>"
    else:
        # Assistant response is the label. Skip if it's missing.
        print(f"Skipping row due to missing assistant response: {row.name}")
        return None

    return {"text": formatted_chat}

def main(input_file, output_file):
    """
    Reads a CSV file, converts it to Llama 3.2 chat format,
    and saves it as a JSONL file.

    Args:
        input_file (str): Path to the input CSV file.
        output_file (str): Path to the output JSONL file.
    """
    try:
        # Load the dataset from a CSV file
        df = pd.read_csv(input_file)
        print(f"Successfully loaded {len(df)} rows from {input_file}")

        # Ensure required columns exist
        required_columns = ['user', 'assistant']
        if not all(col in df.columns for col in required_columns):
            print(f"Error: Input file must contain 'user' and 'assistant' columns.")
            return

        # The 'system' column is optional
        if 'system' not in df.columns:
            print("Warning: 'system' column not found. Proceeding without system prompts.")
            df['system'] = '' # Add an empty system column if it doesn't exist

        # Apply the formatting function to each row
        formatted_data = df.apply(format_for_llama3_2, axis=1).dropna().tolist()

        # Save the formatted data to a JSONL file
        with open(output_file, 'w', encoding='utf-8') as f:
            for item in formatted_data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')

        print(f"\nSuccessfully converted and saved {len(formatted_data)} valid conversations to {output_file}")

        if formatted_data:
            print("\nExample of a formatted conversation:")
            print(formatted_data[0]['text'])
        else:
            print("\nNo valid conversations were found to format.")


    except FileNotFoundError:
        print(f"Error: The file '{input_file}' was not found. Please make sure it's uploaded to your Colab session.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

if __name__ == '__main__':
    # --- Configuration ---
    # 1. Upload your CSV to Colab.
    # 2. Make sure your CSV has 'user' and 'assistant' columns. A 'system' column is optional.
    # 3. Change the file name below to match your uploaded file.

    input_csv_file = '/content/problems.csv'  # <--- CHANGE THIS to your actual file name
    output_jsonl_file = 'formatted_dataset.jsonl' # Name for the output file

    # --- Run the conversion ---
    main(input_csv_file, output_jsonl_file)


Successfully loaded 1 rows from /content/problems.csv

Successfully converted and saved 1 valid conversations to formatted_dataset.jsonl

Example of a formatted conversation:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert cybersecurity analyst and synthetic data engineer. Your mission is to help catalog and generate synthetic datasets for critical cybersecurity challenges across diverse operational environments.

Operational areas:
- **Enterprise**
- **Cloud**
- **Personal**

For each problem you describe, include:
1. **nature**: The specific category or type (e.g., phishing, data exfiltration, misconfiguration).
2. **description**: A concise but comprehensive overview of the issue.
3. **risk_reduction**: Concrete strategies or controls to mitigate the threat.
<|eot_id|><|start_header_id|>user<|end_header_id|>

 Generate new cybersecurity problems.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

[{"area": "Phishing Attack", "nature": "credential_

In [11]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="formatted_dataset.jsonl", split="train")
print(dataset)


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 1
})


In [13]:
dataset[0]

{'text': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are an expert cybersecurity analyst and synthetic data engineer. Your mission is to help catalog and generate synthetic datasets for critical cybersecurity challenges across diverse operational environments.\n\nOperational areas:\n- **Enterprise**\n- **Cloud**\n- **Personal**\n\nFor each problem you describe, include:\n1. **nature**: The specific category or type (e.g., phishing, data exfiltration, misconfiguration).\n2. **description**: A concise but comprehensive overview of the issue.\n3. **risk_reduction**: Concrete strategies or controls to mitigate the threat.\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n Generate new cybersecurity problems.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n[{"area": "Phishing Attack", "nature": "credential_harvesting", "description": "Attackers send emails impersonating trusted entities to trick users into submitting their login credentials on fake web

In [14]:
dataset[0]["text"]

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are an expert cybersecurity analyst and synthetic data engineer. Your mission is to help catalog and generate synthetic datasets for critical cybersecurity challenges across diverse operational environments.\n\nOperational areas:\n- **Enterprise**\n- **Cloud**\n- **Personal**\n\nFor each problem you describe, include:\n1. **nature**: The specific category or type (e.g., phishing, data exfiltration, misconfiguration).\n2. **description**: A concise but comprehensive overview of the issue.\n3. **risk_reduction**: Concrete strategies or controls to mitigate the threat.\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n Generate new cybersecurity problems.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n[{"area": "Phishing Attack", "nature": "credential_harvesting", "description": "Attackers send emails impersonating trusted entities to trick users into submitting their login credentials on fake websites.", 

<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

In [15]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

num_proc must be <= 1. Reducing num_proc to 1 for dataset of size 1.


Unsloth: Tokenizing ["text"]:   0%|          | 0/1 [00:00<?, ? examples/s]

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs.

In [16]:
## Donot use this cell or comment out it for other file than problem csv . As the user input of problems csv is very repitetive. I donot want to overfit on user input.
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

num_proc must be <= 1. Reducing num_proc to 1 for dataset of size 1.


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

We verify masking is actually done:

In [17]:
tokenizer.decode(trainer.train_dataset[0]["input_ids"])

'<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are an expert cybersecurity analyst and synthetic data engineer. Your mission is to help catalog and generate synthetic datasets for critical cybersecurity challenges across diverse operational environments.\n\nOperational areas:\n- **Enterprise**\n- **Cloud**\n- **Personal**\n\nFor each problem you describe, include:\n1. **nature**: The specific category or type (e.g., phishing, data exfiltration, misconfiguration).\n2. **description**: A concise but comprehensive overview of the issue.\n3. **risk_reduction**: Concrete strategies or controls to mitigate the threat.\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n Generate new cybersecurity problems.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n[{"area": "Phishing Attack", "nature": "credential_harvesting", "description": "Attackers send emails impersonating trusted entities to trick users into submitting their login credentials on 

In [19]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[0]["labels"]])

'                                                                                                                                [{"area": "Phishing Attack", "nature": "credential_harvesting", "description": "Attackers send emails impersonating trusted entities to trick users into submitting their login credentials on fake websites.", "risk_reduction": ["Implement multi-factor authentication (MFA) to reduce impact of stolen credentials", "Deploy advanced email filtering and anti-phishing tools", "Conduct regular user training and simulated phishing exercises"]}, {"area": "Phishing Attack", "nature": "spear_phishing", "description": "Highly targeted phishing emails crafted using personal information to deceive specific individuals into revealing sensitive data or executing malicious actions.", "risk_reduction": ["Use threat intelligence to identify and block targeted phishing campaigns", "Educate employees on recognizing social engineering tactics", "Enforce strict verification procedur

We can see the System and Instruction prompts are successfully masked!
***
only for problems.csv
***

In [20]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
6.779 GB of memory reserved.


In [21]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1 | Num Epochs = 60 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
1,1.513500
2,1.513500
3,1.486600
4,1.373700
5,1.205300
6,1.005700
7,0.774200
8,0.551100
9,0.351500
10,0.199800


Unsloth: Will smartly offload gradients to save VRAM!


In [22]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

44.8572 seconds used for training.
0.75 minutes used for training.
Peak reserved memory = 6.779 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 45.987 %.
Peak reserved memory for training % of max memory = 0.0 %.


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!



We use `min_p = 0.1` and `temperature = 1.5`. Read this [Tweet](https://x.com/menhguin/status/1826132708508213629) for more information on why.

Just for checking the inference for problem

In [35]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Generate new cybersecurity problems"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 64, use_cache = True,
                         temperature = 1.5, min_p = 0.1)
tokenizer.batch_decode(outputs)

['<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nGenerate new cybersecurity problems<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n[{"area": "Phishing Attack", "question": ["how to identify and block phishing emails on user devices?", "Phishing Attack: Attackers send emails impersonating trusted entities to trick users into submitting sensitive information on fake websites.", "Countering Phishing Attack: Implement multi-factor authentication (MFA) to']

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("lora_model")  # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Describe a tall tower in the capital of France."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

The Eiffel Tower, located in the heart of Paris, stands tall among the city's historic and cultural landmarks. This iron structure, standing at an impressive 324 meters high, offers breathtaking views of the City of Light's iconic landscape. The Eiffel Tower was built for the 1889 World's Fair and has since become a symbol of French engineering and culture.<|eot_id|>


You can also use Hugging Face's `AutoModelForPeftCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

In [ ]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("lora_model")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False:
    model.save_pretrained("model")
    tokenizer.save_pretrained("model")
if False:
    model.push_to_hub("hf/model", token = "")
    tokenizer.push_to_hub("hf/model", token = "")


### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "", # Get a token at https://huggingface.co/settings/tokens
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp or a UI based system like Jan or Open WebUI. You can install Jan [here](https://github.com/janhq/jan) and Open WebUI [here](https://github.com/open-webui/open-webui)

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
